In [103]:
from utils import create_new_dataset_with_ipcw_weights, create_surv_data, train_test_split_into_df, get_Nbi, ipc_weighted_mse, inf_JK_bagged_variance, create_train_test_data
from lifelines import KaplanMeierFitter, WeibullAFTFitter
from sksurv.metrics import concordance_index_ipcw
from sksurv.util import Surv
from sklearn.model_selection import train_test_split
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

###  Dataset Creation

In [111]:
n = 1000
seed = 42
tau = 12

params = { 'shape_weibull': 1, 'scale_weibull_base': 10000, 'rate_censoring': 0.01, 
          'b_bloodp': -0.405, 'b_diab': -0.4, 'b_age': -0.05, 'b_bmi': -0.01, 'b_kreat': -0.2, 
          'n': n, 'seed': seed, 'tau': tau}

df_train, df_test, n_events_after_cut_train, portion_censored_after_cut_train = create_train_test_data(params)

print(f'portion_events_after_cut_train: {n_events_after_cut_train/df_train.shape[0] *100}%' )
print(f'portion_no-events_after_cut_train: {round((1-n_events_after_cut_train/df_train.shape[0]-portion_censored_after_cut_train)*100,2)}%' )
print(f'portion_censored_after_cut_train: {round(portion_censored_after_cut_train*100,2)}')

portion_events_after_cut_train: 81.0%
portion_no-events_after_cut_train: 6.57%
portion_censored_after_cut_train: 12.43


In [92]:
## Überprüfung der Verteilungen Train und Test Data

# fig, axes = plt.subplots(1, 2, figsize=(6, 4), sharey=True)
# sns.boxplot(ax=axes[0], x='event', y='time', data=df_train)
# axes[0].set_title('Training Data - Time by Event')
# axes[0].set_xlabel('Event')
# axes[0].set_ylabel('Time')
# sns.boxplot(ax=axes[1], x='event', y='time', data=df_test)
# axes[1].set_title('Test Data - Time by Event')
# axes[1].set_xlabel('Event')
# axes[1].set_ylabel('Time')
# y_min = min(df_train['time'].min(), df_test['time'].min())
# y_max = max(df_train['time'].max(), df_test['time'].max())
# axes[0].set_ylim(y_min, y_max)
# axes[1].set_ylim(y_min, y_max)
# plt.tight_layout()
# plt.show()

# train_event_counts = df_train['event'].value_counts(normalize=True)
# test_event_counts = df_test['event'].value_counts(normalize=True)
# print("Relative Häufigkeiten der Events - Trainingsdaten:")
# print(train_event_counts)
# print("\nRelative Häufigkeiten der Events - Testdaten:")
# print(test_event_counts)

# strg k str u  -  kommentar
# strg k str c  -  kommentar entfernen

In [93]:
## Check der Gewichte und 'survived ?'

# print('tau:', tau)
# print(df_train[df_train['survived']==0].sample(2)[['time', 'event','weights_ipcw','survived', ]])
# print('\n')
# print(df_train[df_train['survived']==1].sample(2)[['time', 'event','weights_ipcw','survived', ]])
# print('\n')
# print(df_train[df_train['survived']==999].sample(2)[['time', 'event','weights_ipcw','survived', ]])
# print('\n')
# print(df_train.columns)


### Weibull Modell 

In [112]:
### Modell fitten
aft = WeibullAFTFitter()
aft.fit(df=df_train.drop(['weights_ipcw', 'survived'], axis=1), duration_col='time', event_col='event')
#aft.print_summary()

### IPCW-MSE berechnen at tau
y_pred = aft.predict_survival_function(df_test.drop(['weights_ipcw', 'survived','time','event'], axis=1), times = tau).iloc[0].tolist()
weibull_mse = ipc_weighted_mse(y_true=df_test['survived'].values, y_pred=y_pred, sample_weight=df_test['weights_ipcw'])
print(f'weibull_ipcw_mse auf test data: {round(weibull_mse,4)}')


### IPCW-C-Index berechnen
ci_ipcw, concordant, discordant, tied_risk, tied_time = concordance_index_ipcw(
    survival_train = Surv.from_arrays(event=df_train['event'], time=df_train['time']),
    survival_test  = Surv.from_arrays(event=df_test['event'], time=df_test['time']),
    estimate       =  -aft.predict_expectation(df_test) )
print("IPCW-C-Index:", round(ci_ipcw,4))


### Pred X_erwartung
X_erwartung = pd.DataFrame({'bmi': [25], 'blood_pressure': [0], 'kreatinkinase': [np.exp(5+1/2)], 'diabetes': [0], 'age': [50]})
y_pred_X_erwartung_weibull = aft.predict_survival_function(X_erwartung, times = tau).iloc[0].tolist()

print(f'Prediction für X_erwartung zum überleben am Zeitpunkt tau={tau}: {y_pred_X_erwartung_weibull[0]:.4f}')


weibull_ipcw_mse auf test data: 0.0679
IPCW-C-Index: 0.6921
Prediction für X_erwartung zum überleben am Zeitpunkt tau=12: 0.9498


### Random Forest Modell

In [95]:
X_train = df_train.drop(['time', 'event', 'weights_ipcw', 'survived'], axis=1)
y_train = df_train['survived']
bootstrap_weights = df_train['weights_ipcw']

In [96]:
# # HYperparameter Tuning
# param_dist = {
#     'n_estimators': [int(x) for x in np.linspace(start=X_train.shape[0], stop=X_train.shape[0]*10, num=10)],
#     'max_features': [ 'sqrt', 'log2'],
#     'max_depth': [int(x) for x in np.linspace(1,10 , num=10)] + [None],
#     'min_samples_split':[int(x) for x in np.linspace(1,10 , num=10)],
#     'bootstrap': [True]
# }

# rf = RandomForestClassifier(random_state=seed)
# rf_random = RandomizedSearchCV(estimator=rf, param_distributions=param_dist, n_iter=100, cv=5, verbose=3, random_state=seed, n_jobs=-1)
# rf_random.fit(X_train, y_train, sample_weight=bootstrap_weights)

# print(rf_random.best_params_)
# best_rf = rf_random.best_estimator_

# y_pred = best_rf.predict_proba(df_test.drop(['weights_ipcw', 'survived','time','event'], axis=1))[:,1]
# best_rf_mse = ipc_weighted_mse(y_true=df_test['survived'].values, y_pred=y_pred, sample_weight=df_test['weights_ipcw'])

# print(f'best_rf_ipcw_mse auf test data: {round(best_rf_mse,4)}')

In [116]:
params = {
    'n_estimators': X_train.shape[0]*2,
    'max_depth': 3,
    'min_samples_split': 5,
    'max_features': 'log2',
    'random_state': seed,
    'bootstrap': True
}

clf = RandomForestClassifier(**params)
clf.fit(X_train, y_train, sample_weight=bootstrap_weights)

y_pred = clf.predict_proba(df_test.drop(['weights_ipcw', 'survived','time','event'], axis=1))[:,1]
rf_mse = ipc_weighted_mse(y_true=df_test['survived'].values, y_pred=y_pred, sample_weight=df_test['weights_ipcw'])
print(f'rf_ipcw_mse auf test data: {round(rf_mse,4)}')


y_pred_X_erwartung = clf.predict_proba(X_erwartung)[:,1]
print(f'Prediction für X_erwartung zum überleben am Zeitpunkt tau={tau}: {y_pred_X_erwartung[0]:.4f}')

rf_ipcw_mse auf test data: 0.0688
Prediction für X_erwartung zum überleben am Zeitpunkt tau=12: 0.9474


0.9473977392852285

#### Infinetesimal Jackknife

In [114]:
nbi = get_Nbi(clf.estimators_samples_)
tnb = np.array([tree.predict_proba(X_erwartung.values)[:, 1][0] for tree in clf.estimators_]) 

rf_std = np.std(tnb)
print(f'rf_std: {rf_std:.4f}')

biased_var_estimate, bias_correction = inf_JK_bagged_variance(N_bi=nbi, T_N_b=tnb,weights=bootstrap_weights)
unbiased_var_estimate = biased_var_estimate - bias_correction
print(f'Unbiased std_ijk Estimate: {np.sqrt(unbiased_var_estimate):.8f}')


rf_std: 0.0301
Unbiased std_ijk Estimate: 0.01163957
